# AND-02 · Structured concurrency: the Job tree, cancellation, exceptions

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/AND-02.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: AND-02
title: "Structured concurrency: the Job tree, cancellation, exceptions"
track: android
date: 2026-10-13
code: ""
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "A coroutine must belong to a scope: a parent waits for its children, cancelling the parent cancels them, a failing child cancels its siblings unless the parent is supervised, and cancellation only happens at suspension points (never swallow CancellationException); viewModelScope is supervised and is cancelled when the ViewModel is cleared."
tags: [AND-02, android, kotlin, coroutines, structured-concurrency, cancellation, supervisorjob, viewmodelscope]
pending:
  - "Run the four experiments in StructuredLab.kt and compare with the predictions below"
  - "MQ338 in my own words"
  - "English (decouple, cohesive) and the Mobile Lead rule"
```

## Goal
Know who **owns** a coroutine, what happens when something goes wrong, and how to stop work cleanly.

## The problem
```kotlin
GlobalScope.launch {                 // fire and forget
    val data = api.load()
    textView.text = data             // the screen may be gone by now
}
```
The screen closes but the work keeps running: it can touch a dead screen, leak memory, and its errors vanish. **Structured concurrency** (Roman Elizarov, 2018): a coroutine must belong to a **scope**.

## The tree: four rules
Every coroutine is a child of the scope that launched it, so they form a tree of `Job`s. Kitchen picture: a scope is a **table**, each coroutine is an **order**.

| Rule | What happens |
|---|---|
| **1. Wait** | A table is not finished until all its orders are |
| **2. Cancel goes down** | Cancel the table and every order is cancelled |
| **3. Failure goes up** | One order fails → its table is cancelled → the other orders are cancelled → the exception goes to whoever opened the table |
| **4. Supervisor** | At a supervised table (`SupervisorJob`, `supervisorScope`) rule 3 is off: a failed order does not cancel its siblings or the table |

## Cancellation is cooperative
`cancel()` raises a flag. The coroutine only notices at its next **suspension point** (`delay`, `withContext`, `await`...), where it throws `CancellationException`.
- **A tight loop that never suspends ignores cancel.** Call `ensureActive()` (or loop on `isActive`).
- **Never swallow it.** `catch (e: Exception)` also catches `CancellationException`, and the coroutine carries on. Rethrow it: `catch (e: CancellationException) { throw e }`.
- **Cleanup** goes in `finally`; if the cleanup suspends, use `withContext(NonCancellable)`.
`CancellationException` is normal, not an error: it is how cancellation travels.

## Exceptions
- `launch`: the exception goes up to the parent at once; unhandled, the app crashes.
- `async`: the exception is stored and rethrown at `await()`, **and** (in a normal scope) it also cancels the parent at once.
- `try { launch { ... } } catch` does not work: the exception happens later, in another coroutine. Put the `try/catch` **inside** the lambda.
- `CoroutineExceptionHandler` is a last-resort logger for top-level coroutines; it does not stop the cancellation.

| | `coroutineScope { }` | `supervisorScope { }` |
|---|---|---|
| A child fails | siblings cancelled, exception rethrown to the caller | siblings keep running |
| Use it for | pieces of one result (all or nothing) | independent pieces |

## Job vs SupervisorJob
A normal `Job`: a failing child cancels its parent, so its siblings. A `SupervisorJob`: a failure is that child's own problem. **Four mistakes:** (1) `launch(SupervisorJob()) { }` makes the `SupervisorJob` the *parent*, so the coroutine leaves your scope; (2) supervision is one level deep; (3) it does not stop crashes, handle expected errors inside the coroutine; (4) `async` under a supervisor needs a `try/catch` around each `await()`.

## `viewModelScope` (MQ338)
`SupervisorJob() + Dispatchers.Main.immediate`. It is cancelled when the ViewModel is **cleared** (the screen is finished for good, **not** on rotation); every child then gets a `CancellationException` at its next suspension point, and Retrofit's suspend calls cancel the HTTP call. It is supervised so one failed request does not cancel the others on the screen; expected failures still need a `try/catch` inside.

## The scope catalog
| Scope | Dies when | Use it for |
|---|---|---|
| `GlobalScope` | never | almost nothing (it leaks) |
| `viewModelScope` | the ViewModel is cleared | loading data for a screen |
| `lifecycleScope` | the lifecycle is destroyed | UI work (for Flow: `repeatOnLifecycle(STARTED)`) |
| `rememberCoroutineScope()` | the composable leaves | click handlers that start work |
| `LaunchedEffect(key)` | it leaves the composition, or the key changes | work tied to a key |
| `coroutineScope { }` | all children finish | pieces of one result |
| `supervisorScope { }` | all children finish | independent pieces |
| your own `CoroutineScope(SupervisorJob() + ...)` | you call `cancel()` | work that outlives screens |
| `CoroutineWorker` | the work finishes or stops | work that must survive the app being killed |

## From the power button to a scope (the hardware view)
Power button → PMIC → SoC reset → Boot ROM → bootloader → kernel → `init` → Zygote → (tap) → `system_server` asks Zygote → `fork()` → `ActivityThread.main()` → the main Looper → your Activity or ViewModel → `viewModelScope.launch { }` → a coroutine (a `Job` that is a child of the scope, plus a state machine) → a dispatcher → a **thread** → the kernel schedules it onto a **core**. The kernel's low-memory killer can end the whole process without a callback, so anything that must finish belongs in WorkManager.

## A runnable model of the four rules
This is **not** real coroutines (they need the kotlinx library). It is a small Python model of the Job tree, so you can run the four rules and watch them. The real experiments are in `StructuredLab.kt` below.

In [ ]:
class Job:
    def __init__(self, name, supervisor=False):
        self.name, self.supervisor = name, supervisor
        self.children, self.parent, self.state = [], None, "active"      # active | done | cancelled | failed
    def launch(self, name):
        child = Job(name); child.parent = self; self.children.append(child); return child
    def cancel(self):                                                   # rule 2: cancel goes down
        if self.state == "active":
            self.state = "cancelled"
            for c in self.children: c.cancel()
    def fail(self):                                                     # rule 3: failure goes up
        self.state = "failed"
        for c in self.children: c.cancel()
        if self.parent and not self.parent.supervisor:
            self.parent.cancel_for_child_failure()
    def cancel_for_child_failure(self):
        if self.state == "active":
            self.state = "cancelled"
            for c in self.children:
                if c.state == "active": c.cancel()
            if self.parent and not self.parent.supervisor: self.parent.cancel_for_child_failure()
    def finish(self):                                                   # rule 1: a parent waits for its children
        assert all(c.state != "active" for c in self.children) or self.state != "active"
        if self.state == "active": self.state = "done"

def states(job): return {c.name: c.state for c in job.children}

normal = Job("scope")
a, b, c = normal.launch("A"), normal.launch("B"), normal.launch("C")
c.fail()
print("normal scope, C fails      :", states(normal), "| scope is", normal.state)
assert states(normal) == {"A": "cancelled", "B": "cancelled", "C": "failed"} and normal.state == "cancelled"

supervised = Job("scope", supervisor=True)
a, b, c = supervised.launch("A"), supervised.launch("B"), supervised.launch("C")
c.fail()
a.finish(); b.finish()
print("supervised scope, C fails  :", states(supervised), "| scope is", supervised.state)
assert states(supervised) == {"A": "done", "B": "done", "C": "failed"} and supervised.state == "active"

tree = Job("scope"); x, y = tree.launch("X"), tree.launch("Y"); y.launch("Y1")
tree.cancel()
print("cancel the parent          :", states(tree), "| Y1 is", y.children[0].state)
assert states(tree) == {"X": "cancelled", "Y": "cancelled"} and y.children[0].state == "cancelled"

## The lab: `StructuredLab.kt` (in the scratch project; not run here)
Needs `kotlinx-coroutines`. Four experiments: (1) `coroutineScope` with a failing child, (2) `supervisorScope` with a failing child and a handler, (3) a loop that ignores cancel, then `ensureActive()`, (4) `catch (e: Exception)` swallowing `CancellationException`.

**Predictions (not yet confirmed by me):**
| Experiment | Expected |
|---|---|
| 1 | After 100 ms C throws; A and B print "stopped" but never "finished"; then "caught outside: C failed" |
| 2 | "handler got: C failed" at about 100 ms; A and B both finish at about 500 ms; then "scope done" |
| 3 | Despite `cancel()` at 100 ms, "loop finished anyway" appears after about 1 second |
| 4 | "step 0", "swallowed: CancellationException" several times, then "kept going after cancel!" |

Fixes: `ensureActive()` as the first line in the loop (3); `catch (e: CancellationException) { throw e }` (4).

## MQ338 · How does `viewModelScope` cancel when a screen closes?
`viewModelScope` is built on a `SupervisorJob`. When the ViewModel is cleared (the screen is finished for good, not on rotation), its scope is cancelled, so every child coroutine receives a `CancellationException` at its next suspension point; suspend calls such as Retrofit's cancel their work, and `finally` blocks run. (Answer from the tutor: write your own version.)

## English and role lens
Words: **decouple** (reduce dependency), **cohesive** (parts belong together). Grammar: past simple vs present perfect.
✏️ Three sentences with both words. ✏️ **Mobile Lead lens:** one rule you would put in the team guide about coroutines (for example: *no `GlobalScope`; every coroutine has an owner*), and why.

## Next
Run the lab and compare it with the predictions. Next Android sessions: AND-03 (dispatchers) on Tue 20 Oct, then AND-04 (Flow internals) on Tue 27 Oct.